# LLaMA3.2-3B Sentiment Pipeline (Gold Commodity Eval)

This notebook runs three stages:
1. **Zero-shot baseline** with `unsloth/Llama-3.2-3B-Instruct` on the Gold Commodity dataset.
2. **qLoRA fine-tuning** of `unsloth/Llama-3.2-3B-Instruct` on full FinancialPhraseBank, then evaluation on Gold Commodity.
3. **Random-search hyperparameter tuning** for LLaMA qLoRA, then final re-evaluation on Gold Commodity.

All stage metrics are logged locally for comparison without W&B.

In [8]:
import getpass

# # Ask once, so the token is not stored in plain text in the notebook
GH_TOKEN = getpass.getpass("GitHub token: ")

In [ ]:
%%capture
!pip install -U unsloth transformers==4.56.2 trl==0.22.2 peft bitsandbytes accelerate datasets scikit-learn matplotlib pandas python-dotenv

In [10]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
%%capture
!pip install -U unsloth transformers==4.56.2 trl==0.22.2 peft bitsandbytes accelerate datasets scikit-learn matplotlib pandas python-dotenv

## Device Check


In [12]:
import subprocess
import torch

print("=== nvidia-smi (if available) ===")
try:
    res = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if res.stdout.strip():
        print(res.stdout)
    else:
        print(res.stderr.strip() or "nvidia-smi returned no output")
except FileNotFoundError:
    print("nvidia-smi not found on this machine.")

if torch.cuda.is_available():
    device_type = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device_type = "mps"
else:
    device_type = "cpu"

print(f"Detected runtime device: {device_type}")
if device_type == "cuda":
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")

=== nvidia-smi (if available) ===
Fri Apr  3 12:14:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P0             29W /   70W |     107MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------

## 1. Setup and Reproducibility

In [13]:
import os
import re
import gc
import math
import time
import random
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def load_dotenv_walk(start: Path) -> bool:
    for parent in [start, *start.parents]:
        env_file = parent / '.env'
        if env_file.exists():
            load_dotenv(env_file, override=False)
            print(f'Loaded .env from: {parent}')
            return True
    return False

_ = load_dotenv_walk(Path(os.getcwd()))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

print(f'Device: {DEVICE}')
print(f'DType : {DTYPE}')

Device: cuda
DType : torch.bfloat16


## 2. W&B + Dataset Loading (Gold + FinancialPhraseBank)

In [16]:
WANDB_PROJECT = os.getenv('WANDB_PROJECT', 'is469-sentiment-comparison')
WANDB_ENTITY = os.getenv('WANDB_ENTITY', None)
WANDB_GROUP = os.getenv('WANDB_GROUP', 'gold-commodity-side-by-side')
print(f"W&B destination -> entity={WANDB_ENTITY}, project={WANDB_PROJECT}, group={WANDB_GROUP}")

# Load Gold dataset for evaluation
gold_ds = load_dataset('SaguaroCapital/sentiment-analysis-in-commodity-market-gold')
split_name = 'test' if 'test' in gold_ds else 'train'
gold_raw_df = gold_ds[split_name].to_pandas()

# Detect columns - Gold dataset has 'News' and 'Price Sentiment'
sentence_col = None
label_col = None

if 'News' in gold_raw_df.columns and 'Price Sentiment' in gold_raw_df.columns:
    sentence_col = 'News'
    label_col = 'Price Sentiment'
else:
    sentence_col = next((c for c in ['sentence', 'text', 'content', 'headline'] if c in gold_raw_df.columns), None)
    label_col = next((c for c in ['label_text', 'label', 'sentiment', 'target'] if c in gold_raw_df.columns), None)

if sentence_col is None or label_col is None:
    raise ValueError(f'Could not detect sentence/label columns in dataset columns={list(gold_raw_df.columns)}')

labels = gold_raw_df[label_col]
# Map Gold sentiment values to standard labels
label_text = labels.astype(str).str.strip().str.lower()
# Map Gold price sentiment to sentiment labels
sentiment_map = {
    'negative': 'negative',
    'neutral': 'neutral', 
    'positive': 'positive',
    'down': 'negative',
    'constant': 'neutral',
    'up': 'positive',
}
label_text = label_text.map(lambda x: sentiment_map.get(x, x))

gold_eval_df = pd.DataFrame({
    'sentence': gold_raw_df[sentence_col].astype(str),
    'label_text': label_text,
})

print(f'Gold eval samples: {len(gold_eval_df)}')
print(gold_eval_df['label_text'].value_counts())

W&B destination -> entity=None, project=is469-sentiment-comparison, group=gold-commodity-side-by-side
Gold eval samples: 2114
label_text
positive    881
negative    764
none        394
neutral      75
Name: count, dtype: int64


In [17]:
# Download FinancialPhraseBank (75% agreement)
!curl -L "https://raw.githubusercontent.com/maxwellsarpong/NLP-financial-text-processing-dataset/master/Sentences_75Agree.txt" -o Sentences_75Agree.txt

fpb_df = pd.read_csv(
    'Sentences_75Agree.txt',
    sep='@',
    header=None,
    names=['sentence', 'label_text'],
    engine='python',
    encoding='latin-1',
)
fpb_df['sentence'] = fpb_df['sentence'].str.strip()
fpb_df['label_text'] = fpb_df['label_text'].str.strip().str.lower()

print(f'FPB full samples: {len(fpb_df)}')
print(fpb_df['label_text'].value_counts())

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  452k  100  452k    0     0   470k      0 --:--:-- --:--:-- --:--:--  470k
FPB full samples: 3453
label_text
neutral     2146
positive     887
negative     420
Name: count, dtype: int64


In [ ]:
# Download FinancialPhraseBank (75% agreement)
!curl -L "https://raw.githubusercontent.com/maxwellsarpong/NLP-financial-text-processing-dataset/master/Sentences_75Agree.txt" -o Sentences_75Agree.txt

fpb_df = pd.read_csv(
    'Sentences_75Agree.txt',
    sep='@',
    header=None,
    names=['sentence', 'label_text'],
    engine='python',
    encoding='latin-1',
)
fpb_df['sentence'] = fpb_df['sentence'].str.strip()
fpb_df['label_text'] = fpb_df['label_text'].str.strip().str.lower()

print(f'FPB full samples: {len(fpb_df)}')
print(fpb_df['label_text'].value_counts())

## 3. Shared Helpers (Prompting, Batched Inference, Metrics, Logging)

In [18]:
from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    matthews_corrcoef,
    confusion_matrix,
)
from transformers import EarlyStoppingCallback

LABELS = ['negative', 'neutral', 'positive']
LABEL_SET = set(LABELS)

def make_prompt(sentence: str) -> str:
    return (
        'You are a financial sentiment classifier.\n'
        'Return exactly one label: negative, neutral, or positive.\n\n'
        f'Sentence: {sentence}\n'
        'Label:'
    )

def parse_label(text: str) -> str:
    t = text.strip().lower()
    m = re.search(r'(negative|neutral|positive)', t)
    return m.group(1) if m else 'neutral'

@torch.inference_mode()
def batched_generate_labels(model, tokenizer, sentences, batch_size=32, max_new_tokens=3):
    preds = []
    tokenizer.padding_side = 'left'
    prompts = [make_prompt(s) for s in sentences]

    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i : i + batch_size]
        enc = tokenizer(
            chunk,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=512,
        ).to(model.device)

        out = model.generate(
            **enc,
            do_sample=False,
            temperature=None,
            top_p=None,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )

        gen_tokens = out[:, enc['input_ids'].shape[1]:]
        texts = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
        preds.extend([parse_label(t) for t in texts])

    return preds

def evaluate_predictions(true_labels, pred_labels):
    true_labels = [x.strip().lower() for x in true_labels]
    pred_labels = [x.strip().lower() if x.strip().lower() in LABEL_SET else 'neutral' for x in pred_labels]

    metrics = {
        'macro_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='macro', zero_division=0),
        'micro_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='micro', zero_division=0),
        'weighted_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='weighted', zero_division=0),
        'f1_negative': f1_score(true_labels, pred_labels, labels=['negative'], average='macro', zero_division=0),
        'f1_neutral': f1_score(true_labels, pred_labels, labels=['neutral'], average='macro', zero_division=0),
        'f1_positive': f1_score(true_labels, pred_labels, labels=['positive'], average='macro', zero_division=0),
        'accuracy': accuracy_score(true_labels, pred_labels),
        'mcc': matthews_corrcoef(true_labels, pred_labels),
    }

    metrics['classification_report'] = classification_report(
        true_labels, pred_labels, labels=LABELS, zero_division=0
    )
    metrics['confusion_matrix'] = confusion_matrix(true_labels, pred_labels, labels=LABELS)
    return metrics

def print_metrics(title, metrics):
    print('=' * 80)
    print(title)
    print('=' * 80)
    print(metrics['classification_report'])
    print(f"Macro F1   : {metrics['macro_f1']:.4f}")
    print(f"Micro F1   : {metrics['micro_f1']:.4f}")
    print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
    print(f"Accuracy   : {metrics['accuracy']:.4f}")
    print(f"MCC        : {metrics['mcc']:.4f}")
    print(f"F1 neg     : {metrics['f1_negative']:.4f}")
    print(f"F1 neu     : {metrics['f1_neutral']:.4f}")
    print(f"F1 pos     : {metrics['f1_positive']:.4f}")

def log_run_to_wandb(run_name, config_dict, metrics):
    # W&B logging has been disabled
    print(f"[W&B Disabled] Would have logged run: {run_name}")

def create_eval_metric_fn():
    """Create an evaluation metric function for use with early stopping."""
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        # For SFTTrainer, we get loss-based metrics
        return {'loss': preds if isinstance(preds, float) else 0.0}
    return compute_metrics

## 4. Baseline: LLaMA3.2-3B-Instruct (Non-Finetuned) on Gold Commodity

In [22]:
BASE_MODEL_NAME = 'unsloth/Llama-3.2-3B-Instruct'

baseline_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
if baseline_tokenizer.pad_token is None:
    baseline_tokenizer.pad_token = baseline_tokenizer.eos_token

baseline_model, baseline_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(baseline_model)
baseline_model.eval()

baseline_preds = batched_generate_labels(
    baseline_model,
    baseline_tokenizer,
    gold_eval_df['sentence'].tolist(),
    batch_size=64,
    max_new_tokens=3,
)

baseline_metrics = evaluate_predictions(gold_eval_df['label_text'].tolist(), baseline_preds)
print_metrics('LLaMA3.2-3B-Instruct (Zero-shot) on Gold Commodity', baseline_metrics)

log_run_to_wandb(
    run_name='llama3.2-3b-instruct_zero_shot_gold',
    config_dict={
        'model_name': BASE_MODEL_NAME,
        'stage': 'baseline_zero_shot',
        'dataset_eval': 'gold-commodity-sentiment-eval',
        'quantization': '4bit',
        'batch_size': 64,
    },
    metrics=baseline_metrics,
)

==((====))==  Unsloth 2026.4.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
LLaMA3.2-3B-Instruct (Zero-shot) on Gold Commodity
              precision    recall  f1-score   support

    negative       0.72      0.85      0.78       764
     neutral       0.09      0.73      0.16        75
    positive       0.83      0.56      0.67       881

   micro avg       0.57      0.70      0.63      1720
   macro avg       0.55      0.72      0.54      1720
weighted avg       0.75      0.70      0.70      1720

Macro F1   : 0.5366
Micro F1   : 0.6260
Weighted F1: 0.6968
Accuracy   : 0.5676
MCC        : 0.4348
F1 neg     : 0.78

## 5. Fine-tune LLaMA3.2-3B-Instruct (Full FPB), Then Evaluate on Gold

In [19]:
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq

LLAMA_MODEL_NAME = 'unsloth/Llama-3.2-3B-Instruct'
LLAMA_ADAPTER_DIR = 'qlora_adapters/llama3.2-3b-fpb-full-default'
MAX_SEQ_LEN = 1024

def make_chat_train_text(tokenizer, sentence, label):
    messages = [
        {
            'role': 'system',
            'content': 'You are a financial sentiment classifier. Reply with one word: negative, neutral, or positive.',
        },
        {'role': 'user', 'content': f'Classify sentiment:\n{sentence}'},
        {'role': 'assistant', 'content': label},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

llama_model, llama_tokenizer = FastLanguageModel.from_pretrained(
    model_name=LLAMA_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

llama_model = FastLanguageModel.get_peft_model(
    llama_model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

# Create train/val split from FPB
train_texts = [make_chat_train_text(llama_tokenizer, s, y) for s, y in zip(fpb_df['sentence'], fpb_df['label_text'])]
split_idx = int(0.8 * len(train_texts))
train_texts_split = train_texts[:split_idx]
eval_texts_split = train_texts[split_idx:]

train_ds = Dataset.from_dict({'text': train_texts_split})
eval_ds = Dataset.from_dict({'text': eval_texts_split})

print(f'Train samples: {len(train_ds)}, Eval samples: {len(eval_ds)}')

trainer = SFTTrainer(
    model=llama_model,
    tokenizer=llama_tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    data_collator=DataCollatorForSeq2Seq(tokenizer=llama_tokenizer),
    packing=False,
    args=SFTConfig(
        output_dir='outputs/llama3.2-3b-fpb-full-default',
        num_train_epochs=3,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        weight_decay=0.01,
        optim='adamw_8bit',
        logging_steps=20,
        save_strategy='steps',
        save_steps=50,
        eval_strategy='steps',
        eval_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model='loss',
        seed=SEED,
        report_to='none',
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.001,
        )
    ],
)

trainer = train_on_responses_only(
    trainer,
    instruction_part='<|start_header_id|>user<|end_header_id|>\n\n',
    response_part='<|start_header_id|>assistant<|end_header_id|>\n\n',
)

print('Starting training with EarlyStopping enabled...')
train_out = trainer.train()
llama_model.save_pretrained(LLAMA_ADAPTER_DIR)
llama_tokenizer.save_pretrained(LLAMA_ADAPTER_DIR)

FastLanguageModel.for_inference(llama_model)
llama_model.eval()

llama_preds = batched_generate_labels(
    llama_model,
    llama_tokenizer,
    gold_eval_df['sentence'].tolist(),
    batch_size=64,
    max_new_tokens=3,
)

llama_metrics = evaluate_predictions(gold_eval_df['label_text'].tolist(), llama_preds)
print_metrics('LLaMA3.2-3B qLoRA (Default Hyperparams) on Gold Commodity', llama_metrics)

log_run_to_wandb(
    run_name='llama3.2-3b-qlora_default_fullfpb_gold',
    config_dict={
        'model_name': LLAMA_MODEL_NAME,
        'adapter_dir': LLAMA_ADAPTER_DIR,
        'stage': 'finetune_default',
        'dataset_train': 'financialphrasebank_75agree_full',
        'dataset_eval': 'gold-commodity-sentiment-eval',
        'lora_r': 16,
        'lora_alpha': 32,
        'lora_dropout': 0.05,
        'epochs': 3,
        'lr': 2e-4,
        'batch_size': 8,
        'grad_accum': 4,
        'train_loss': float(train_out.training_loss),
    },
    metrics=llama_metrics,
)

/tmp/ipykernel_4584/3104957242.py:2: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,762 | Num Epochs = 3 | Total steps = 261
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,0.101500,0.228328
100,0.040900,0.137755
150,0.030700,0.182472
200,0.004900,0.186424
250,0.003900,0.195460


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


LLaMA3.2-3B qLoRA (Default Hyperparams) on Gold Commodity
              precision    recall  f1-score   support

    negative       0.89      0.80      0.84       764
     neutral       0.10      0.68      0.17        75
    positive       0.83      0.85      0.84       881

   micro avg       0.67      0.82      0.74      1720
   macro avg       0.61      0.78      0.62      1720
weighted avg       0.83      0.82      0.81      1720

Macro F1   : 0.6180
Micro F1   : 0.7360
Weighted F1: 0.8133
Accuracy   : 0.6675
MCC        : 0.5559
F1 neg     : 0.8449
F1 neu     : 0.1683
F1 pos     : 0.8407
[W&B Disabled] Would have logged run: llama3.2-3b-qlora_default_fullfpb_gold


## 6. Random Search Tuning (LLaMA qLoRA) + Final Evaluation on Gold

This stage uses a small random search over qLoRA hyperparameters with early stopping to keep tuning fast on GPU clusters.

In [ ]:
import random
from pathlib import Path

RANDOM_SEARCH_TRIALS = int(os.getenv('RANDOM_SEARCH_TRIALS', 3))
RANDOM_SEARCH_SEED = int(os.getenv('RANDOM_SEARCH_SEED', SEED))
rng = random.Random(RANDOM_SEARCH_SEED)

BEST_RANDOM_ADAPTER_DIR = Path('experiments/sentiment_agent/saved_models/llama_random_search_best')
BEST_RANDOM_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

search_space = {
    'lora_r': [8, 16],
    'lora_alpha': [16, 32, 64],
    'lora_dropout': [0.0, 0.05, 0.1],
    'learning_rate': [1e-4, 2e-4, 3e-4],
    'batch_size': [4],
    'grad_accum': [2, 4],
    'epochs': [1, 2],
}

def sample_random_params():
    return {
        'lora_r': rng.choice(search_space['lora_r']),
        'lora_alpha': rng.choice(search_space['lora_alpha']),
        'lora_dropout': rng.choice(search_space['lora_dropout']),
        'learning_rate': rng.choice(search_space['learning_rate']),
        'batch_size': rng.choice(search_space['batch_size']),
        'grad_accum': rng.choice(search_space['grad_accum']),
        'epochs': rng.choice(search_space['epochs']),
    }

best_random_params = None
best_random_score = -1.0
best_random_metrics = None
best_random_trial = -1

for trial_idx in range(RANDOM_SEARCH_TRIALS):
    params = sample_random_params()
    print(f'Random search trial {trial_idx + 1}/{RANDOM_SEARCH_TRIALS}: {params}')

    model, tok = FastLanguageModel.from_pretrained(
        model_name=LLAMA_MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=params['lora_r'],
        lora_alpha=params['lora_alpha'],
        lora_dropout=params['lora_dropout'],
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=SEED,
    )

    trial_out_dir = f'outputs/random_search/trial_{trial_idx}'
    trial_trainer = SFTTrainer(
        model=model,
        tokenizer=tok,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LEN,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tok),
        packing=False,
        args=SFTConfig(
            output_dir=trial_out_dir,
            num_train_epochs=params['epochs'],
            per_device_train_batch_size=params['batch_size'],
            gradient_accumulation_steps=params['grad_accum'],
            learning_rate=params['learning_rate'],
            lr_scheduler_type='cosine',
            warmup_ratio=0.03,
            weight_decay=0.01,
            optim='adamw_8bit',
            logging_steps=30,
            save_strategy='steps',
            save_steps=30,
            eval_strategy='steps',
            eval_steps=30,
            load_best_model_at_end=True,
            metric_for_best_model='loss',
            seed=SEED,
            report_to='none',
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
        ),
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=2,
                early_stopping_threshold=0.001,
            )
        ],
    )

    trial_trainer = train_on_responses_only(
        trial_trainer,
        instruction_part='<|start_header_id|>user<|end_header_id|>\n\n',
        response_part='<|start_header_id|>assistant<|end_header_id|>\n\n',
    )

    try:
        _ = trial_trainer.train()
    except torch.cuda.OutOfMemoryError:
        print(f'Random search trial {trial_idx + 1} hit CUDA OOM during training; skipping')
        del trial_trainer, model, tok
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

    FastLanguageModel.for_inference(model)
    model.eval()

    del trial_trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    try:
        trial_preds = batched_generate_labels(
            model,
            tok,
            gold_eval_df['sentence'].tolist(),
            batch_size=4,
            max_new_tokens=2,
        )
        trial_metrics = evaluate_predictions(gold_eval_df['label_text'].tolist(), trial_preds)
    except torch.cuda.OutOfMemoryError:
        print(f'Random search trial {trial_idx + 1} hit CUDA OOM during eval; skipping')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        del model, tok
        gc.collect()
        continue

    print(f'Random search trial {trial_idx + 1} macro_f1: {trial_metrics["macro_f1"]:.4f}')
    if trial_metrics['macro_f1'] > best_random_score:
        best_random_score = trial_metrics['macro_f1']
        best_random_params = params
        best_random_metrics = trial_metrics
        best_random_trial = trial_idx + 1

        model.save_pretrained(BEST_RANDOM_ADAPTER_DIR)
        tok.save_pretrained(BEST_RANDOM_ADAPTER_DIR)
        (BEST_RANDOM_ADAPTER_DIR / 'random_search_meta.json').write_text(
            str({
                'best_random_trial': best_random_trial,
                'best_random_score': float(best_random_score),
                'best_random_params': best_random_params,
                'base_model_name': LLAMA_MODEL_NAME,
            }),
            encoding='utf-8',
        )
        print(f'Saved current best adapter to: {BEST_RANDOM_ADAPTER_DIR}')

    del model, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if best_random_params is None:
    raise RuntimeError('Random search did not complete any successful trials.')

print('Best random search trial:', best_random_trial)
print('Best random search macro F1:', best_random_score)
print('Best random search params:', best_random_params)
print(f'Best adapter directory: {BEST_RANDOM_ADAPTER_DIR}')

Random search trial 1/3: {'lora_r': 8, 'lora_alpha': 16, 'lora_dropout': 0.1, 'learning_rate': 0.0002, 'batch_size': 4, 'grad_accum': 2, 'epochs': 1}
==((====))==  Unsloth 2026.4.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/2762 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/691 [00:00<?, ? examples/s]

Map (num_proc=5):   0%|          | 0/2762 [00:00<?, ? examples/s]

...

Filter (num_proc=5):   0%|          | 0/691 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,762 | Num Epochs = 2 | Total steps = 692
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 12,156,928 of 3,224,906,752 (0.38% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
30,0.229900,0.585693
60,0.156900,0.239487
90,0.081700,0.194077
120,0.118200,0.330150
150,0.126800,0.182009
180,0.062900,0.151573
210,0.053000,0.120947
240,0.100500,0.138418
270,0.061800,0.158491


Random search trial 3 macro_f1: 0.6274
Best random search trial: 2
Best random search macro F1: 0.6399334693245767
Best random search params: {'lora_r': 8, 'lora_alpha': 64, 'lora_dropout': 0.1, 'learning_rate': 0.0003, 'batch_size': 4, 'grad_accum': 4, 'epochs': 1}


In [39]:
# Load saved adapter and evaluate only (no retraining)
best = best_random_params
BEST_ADAPTER_DIR = 'experiments/sentiment_agent/saved_models/llama_random_search_best'
FALLBACK_ADAPTER_DIR = LLAMA_ADAPTER_DIR

from pathlib import Path

candidate_paths = [Path(BEST_ADAPTER_DIR), Path(FALLBACK_ADAPTER_DIR)]
load_path = next((p for p in candidate_paths if p.exists()), None)
used_cached_metrics = False

if load_path is not None:
    try:
        print(f'Loading adapter from: {load_path}')
        best_model, best_tok = FastLanguageModel.from_pretrained(
            model_name=str(load_path),
            max_seq_length=MAX_SEQ_LEN,
            dtype=None,
            load_in_4bit=True,
        )
        FastLanguageModel.for_inference(best_model)
        best_model.eval()

        best_preds = batched_generate_labels(
            best_model,
            best_tok,
            gold_eval_df['sentence'].tolist(),
            batch_size=4,
            max_new_tokens=2,
        )

        best_metrics = evaluate_predictions(gold_eval_df['label_text'].tolist(), best_preds)
        print_metrics('LLaMA3.2-3B qLoRA (Loaded Adapter, No Retrain) on Gold Commodity', best_metrics)
    except torch.cuda.OutOfMemoryError:
        print('Adapter load/eval hit CUDA OOM; falling back to cached best_random_metrics.')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if 'best_random_metrics' not in globals() or best_random_metrics is None:
            raise RuntimeError('OOM during load and no cached best_random_metrics available.')
        best_metrics = best_random_metrics
        used_cached_metrics = True
else:
    print('No adapter directory found. Falling back to cached best_random_metrics from search run.')
    if 'best_random_metrics' not in globals() or best_random_metrics is None:
        raise FileNotFoundError(
            'No saved adapter found and no cached best_random_metrics available. Run random search cell first.'
        )
    best_metrics = best_random_metrics
    used_cached_metrics = True

log_run_to_wandb(
    run_name='llama3.2-3b-qlora_random_search_best_fullfpb_gold',
    config_dict={
        'model_name': LLAMA_MODEL_NAME,
        'adapter_dir': str(load_path) if load_path is not None else 'cached_metrics_only',
        'stage': 'evaluate_loaded_random_search_best',
        'dataset_train': 'financialphrasebank_75agree_full',
        'dataset_eval': 'gold-commodity-sentiment-eval',
        **best,
        'random_search_trial': int(best_random_trial),
        'random_search_macro_f1': float(best_random_score),
        'retrained': False,
        'used_cached_metrics': used_cached_metrics,
    },
    metrics=best_metrics,
)

Loading adapter from: qlora_adapters/llama3.2-3b-fpb-full-default
==((====))==  Unsloth 2026.4.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Adapter load/eval hit CUDA OOM; falling back to cached best_random_metrics.
[W&B Disabled] Would have logged run: llama3.2-3b-qlora_random_search_best_fullfpb_gold


## 7. Compact Comparison Table

In [37]:
comparison_df = pd.DataFrame([
    {'model': 'LLaMA3.2-3B-Instruct (zero-shot)', **{k: v for k, v in baseline_metrics.items() if isinstance(v, (int, float, np.floating))}},
    {'model': 'LLaMA3.2-3B qLoRA (default)', **{k: v for k, v in llama_metrics.items() if isinstance(v, (int, float, np.floating))}},
    {'model': 'LLaMA3.2-3B qLoRA (random search best)', **{k: v for k, v in best_metrics.items() if isinstance(v, (int, float, np.floating))}},
])

display_cols = ['model', 'macro_f1', 'micro_f1', 'weighted_f1', 'accuracy', 'mcc', 'f1_negative', 'f1_neutral', 'f1_positive']
display(comparison_df[display_cols].sort_values('macro_f1', ascending=False).reset_index(drop=True))

,model,macro_f1,micro_f1,weighted_f1,accuracy,mcc,f1_negative,f1_neutral,f1_positive
0,LLaMA3.2-3B qLoRA (random search best),0.639933,0.790819,0.834669,0.717124,0.598787,0.884540,0.188862,0.846398
1,LLaMA3.2-3B qLoRA (default),0.617998,0.736046,0.813284,0.667455,0.555860,0.844935,0.168317,0.840743
2,LLaMA3.2-3B-Instruct (zero-shot),0.536604,0.625978,0.696820,0.567644,0.434797,0.781250,0.159190,0.669371


## 8. Placeholder Save of Best Model


In [ ]:
from pathlib import Path
from unsloth import FastLanguageModel

LOAD_BASE_MODEL = 'unsloth/Llama-3.2-3B-Instruct'
LOAD_ADAPTER_DIR = Path('experiments/sentiment_agent/saved_models/llama_random_search_best')

if not LOAD_ADAPTER_DIR.exists():
    raise FileNotFoundError(f'Adapter directory not found: {LOAD_ADAPTER_DIR}')

loaded_model, loaded_tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(LOAD_ADAPTER_DIR),
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(loaded_model)
loaded_model.eval()

print(f'Loaded adapter model from: {LOAD_ADAPTER_DIR}')
print(f'Base model reference: {LOAD_BASE_MODEL}')

## 9. Load Best Adapter Later (Local Repo)

Use this cell in a fresh session to reload the saved random-search best adapter from your local repository.

In [ ]:
from pathlib import Path

EXPORT_DIR = Path('experiments/sentiment_agent/saved_models/random_search_best_placeholder')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
BEST_RANDOM_ADAPTER_DIR = Path('experiments/sentiment_agent/saved_models/llama_random_search_best')

if BEST_RANDOM_ADAPTER_DIR.exists() and any(BEST_RANDOM_ADAPTER_DIR.iterdir()):
    print(f'Best random-search adapter already saved at: {BEST_RANDOM_ADAPTER_DIR}')
elif 'best_model' in globals() and 'best_tok' in globals():
    best_model.save_pretrained(BEST_RANDOM_ADAPTER_DIR)
    best_tok.save_pretrained(BEST_RANDOM_ADAPTER_DIR)
    print(f'Saved best model/tokenizer to: {BEST_RANDOM_ADAPTER_DIR}')
elif 'llama_model' in globals() and 'llama_tokenizer' in globals():
    llama_model.save_pretrained(BEST_RANDOM_ADAPTER_DIR)
    llama_tokenizer.save_pretrained(BEST_RANDOM_ADAPTER_DIR)
    print(f'Saved fallback llama model/tokenizer to: {BEST_RANDOM_ADAPTER_DIR}')
elif 'model' in globals() and 'tokenizer' in globals():
    model.save_pretrained(BEST_RANDOM_ADAPTER_DIR)
    tokenizer.save_pretrained(BEST_RANDOM_ADAPTER_DIR)
    print(f'Saved generic model/tokenizer to: {BEST_RANDOM_ADAPTER_DIR}')
else:
    placeholder = EXPORT_DIR / 'README_PLACEHOLDER.txt'
    placeholder.write_text(
        'No in-memory model object found. Random-search best adapter path is experiments/sentiment_agent/saved_models/llama_random_search_best.',
        encoding='utf-8',
    )
    print(f'Created placeholder file: {placeholder}')

Created placeholder file: experiments/sentiment_agent/saved_models/random_search_best_placeholder/README_PLACEHOLDER.txt


## 8. Cleanup (Optional)

In [35]:
for var_name in ['baseline_model', 'baseline_tokenizer', 'llama_model', 'llama_tokenizer', 'best_model', 'best_tok', 'trainer', 'best_trainer']:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cleanup complete.')

Cleanup complete.
